In [ ]:
import gurobipy as gp
from gurobipy import GRB

In [ ]:
import os
import pandas as pd
directorio=r"C:\Users\melco\OneDrive\Escritorio\pasantia_imple\datos_TEST"
d={}
tuplas={}
B_num={}
p={1:869.434,2:1533.228,3:2430.901,4:3169.766,5:3947.629,6:4690.218,7:5514.84,8:6051.184,9:6693.546,
   10:7264.281,11:8095.383,12:8530.091,13:9088.648,14:9600.03,15:10486.354,16:11616.937,17:12249.156,
   18:12605.908,19:14312.13,20:15063.13,21:16036.058,22:16509.946,23:17643.719,24:18142.476,25:19010.222}
idx_p=gp.tuplelist(p)
maxi=0
sup=list(p.values())
sup=sup[:-1]

# Calcular las medias entre paradas consecutivas
medias_paradas = [(p[i] + p[i+1]) / 2 for i in range(1, 25)]
#print(sup)
#print(len(sup))
#print(medias_paradas)
#print(len(medias_paradas))
#M=max([x1 - x2 for x1, x2 in zip(medias_paradas, sup)])
#print(M)
#print(resultado)
#print(medias_paradas-sup)
print(medias_paradas)
medias_paradas.append(float('inf'))  # Añadir un valor infinito al final para la última parada

for j, archivo in enumerate(os.listdir(directorio)):
    if archivo.endswith('.csv'):
        cont=0
        ruta_archivo = os.path.join(directorio, archivo)
        df = pd.read_csv(ruta_archivo)
        df['Distance'] = df['Distance'].fillna(0)
        df['dist_Acum'] = df['Distance'].cumsum()
        filtered_df = df[df['Speed[m/s]']<=3.03]
        aux=df['dist_Acum'].iloc[-1]
        if aux>=maxi:
            maxi=aux
        k=0
        for i, row in filtered_df.iterrows():
            k=k+1
            dist_acum=row['dist_Acum']
            parada = next((idx + 1 for idx, media in enumerate(medias_paradas) if dist_acum <= media), 25)
            d[(j+1, k,parada)] = dist_acum
            tuplas[(j+1, k)]=dist_acum
            cont=cont+1
            B_num[j+1]=cont
        #print(j)

B = {}

M=maxi
# Iterar sobre el diccionario original
for clave, valor in d.items():
    primer_valor = clave[0]
    if primer_valor not in B:
        B[primer_valor] = {}
    B[primer_valor][clave] = valor
    
B_max=max(list(B_num.values()))
#print(B_max)

idx_tuplas=gp.tuplelist(tuplas)
#print(M)
#print(d)
#print(tuplas)
#print(idx_tuplas)
#print(tuplas)
idx_d=gp.tuplelist(d)

#print(p)
#print(B[1])
#print(B_num)
idx_B=gp.tuplelist(B_num)
#print(idx_B)
#print(idx_d)
S=sum(B_num.values())
#print(S)
print(M)

In [ ]:
m1 = gp.Model()
x=m1.addVars(idx_d, vtype = GRB.BINARY, name="x")
y=m1.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y",ub=1706.222)
z=m1.addVars(idx_tuplas,vtype=GRB.CONTINUOUS, name="z")
m1.setObjective(z.sum("*","*"), GRB.MINIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m1.addConstrs((x.sum(i,"*","*")>=25*0.5 for i in idx_B),"max_base")
m1.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_tuplas ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m1.addConstrs((z[i,j]>=tuplas[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j,k) in idx_d),"cota_inf")
m1.addConstrs((z[i,j]>=-tuplas[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j,k) in idx_d),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m1.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m1.setParam('TimeLimit', 600)
m1.optimize()

In [ ]:
m1.write("test2.lp")
if m1.SolCount > 0:
    # Recuperar los valores de las variables
    vx = m1.getAttr('x', x)
    vy = m1.getAttr('x', y) 
    vz = m1.getAttr('x', z)
    for i,j,k in idx_d:
        if vx[i,j,k] > 0:
            print("Base de datos {} punto elegido {} para la parada {}.".format(i, j,k))
    for i in idx_B:
             print(f"Desplazamiento {vy[i]}") 
    for i,j in idx_tuplas:
        if vz[i,j]!=0:
            print(f"El error del punto {j} con valor {tuplas[i,j]}en la base {i} es {vz[i,j]}")
            

In [ ]:
m = gp.Model()
x=m.addVars(idx_d, vtype = GRB.BINARY, name="x")
y=m.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y",ub=1706.222)
z=m.addVars(idx_tuplas,vtype=GRB.CONTINUOUS, name="z")
m.setObjective(z.sum("*","*"), GRB.MINIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m.addConstrs((x.sum(i,"*","*")>=25*0.9 for i in idx_B),"max_base")
#m.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_tuplas ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m.addConstrs((z[i,j]>=tuplas[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j,k) in idx_d),"cota_inf")
m.addConstrs((z[i,j]>=-tuplas[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j,k) in idx_d),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m.setParam('TimeLimit', 1000)
m.optimize()

In [ ]:
m.write("test3.lp")
if m.SolCount > 0:
    # Recuperar los valores de las variables
    vx = m.getAttr('x', x)
    vy = m.getAttr('x', y) 
    vz = m.getAttr('x', z)
    for i,j,k in idx_d:
        if vx[i,j,k] > 0:
            print("Base de datos {} punto elegido {} para la parada {}.".format(i, j,k))
    for i in idx_B:
             print(f"Desplazamiento {vy[i]}") 
    for i,j in idx_tuplas:
        if vz[i,j]!=0:
            print(f"El error del punto {j} con valor {tuplas[i,j]}en la base {i} es {vz[i,j]}")

In [ ]:
M=2706.222+16248
m3 = gp.Model()
x=m3.addVars(idx_d, vtype = GRB.BINARY, name="x")
y=m3.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y",ub=1706.222)
z=m3.addVars(idx_tuplas,vtype=GRB.CONTINUOUS, name="z")
m3.setObjective(z.sum("*","*"), GRB.MINIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m3.addConstrs((x.sum(i,"*","*")>=25*0.9 for i in idx_B),"max_base")
m3.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_tuplas ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m3.addConstrs((z[i,j]>=tuplas[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j,k) in idx_d),"cota_inf")
m3.addConstrs((z[i,j]>=-tuplas[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j,k) in idx_d),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m3.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m3.setParam('TimeLimit', 1000)
m3.optimize()

In [ ]:
m3.write("test4.lp")
if m3.SolCount > 0:
    # Recuperar los valores de las variables
    vx = m3.getAttr('x', x)
    vy = m3.getAttr('x', y) 
    vz = m3.getAttr('x', z)
    for i,j,k in idx_d:
        if vx[i,j,k] > 0:
            print("Base de datos {} punto elegido {} para la parada {}.".format(i, j,k))
    for i in idx_B:
             print(f"Desplazamiento {vy[i]}") 
    for i,j in idx_tuplas:
        if vz[i,j]!=0:
            print(f"El error del punto {j} con valor {tuplas[i,j]}en la base {i} es {vz[i,j]}")